<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_36_typing_pydantic/note_lesson_36_pydantic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧾 Урок 36 — Typing + Pydantic: новинний агрегатор, крок 1

| Крок | Що робимо |
|---|---|
| 0 | завантажуємо `news_hub` і знімок 168 новин rbc.ua |
| 1 | анотації типів і `mypy` (вправи 1–2) |
| 2 | `NewsItem`: перевірка справжніх новин (вправи 3–4) |
| 3 | власні моделі: `Field`, `Literal`, `field_validator` (вправи 5–6) |
| 4 | весь знімок, JSON, JSON Schema, `TypeAdapter` (вправи 7–9) |
| 5 | знайди помилку: URL рядком (вправа 10) |

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам. Теорія й діфи рефакторингу — у книзі курсу: [Урок 36](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_36/).

---
## 0. Проєкт у ноутбуці

`news_hub` — парсер `parse_rbc_news` зі старого курсу з анотаціями типів (`parser.py`), модель `NewsItem` (`models.py`) і знімок стрічки rbc.ua (`data/rbc_news_snapshot.json`, 168 новин). У Colab клітинка нижче завантажить лише папку уроку і встановить залежності.

In [ ]:
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_36_typing_pydantic"
REPO = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("news_hub"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

try:
    import bs4, mypy, pydantic  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "news_hub/requirements.txt"], check=True)

os.chdir("news_hub")
sys.path.insert(0, os.getcwd())

from collections import Counter
from datetime import time
from typing import Literal

from pydantic import BaseModel, Field, TypeAdapter, ValidationError, field_validator

from news_hub.models import NewsItem, validate_news
from news_hub.parser import RawNews
from news_hub.snapshot import load_snapshot

raw_news = load_snapshot()
print("новин у знімку:", len(raw_news))
print(raw_news[1])

---
## 1. Анотації типів

Анотації — підказки: Python їх не перевіряє, перевіряє `mypy` до запуску.

### Вправа 1

Напиши `count_by_lang(items)` **з анотаціями**: приймає список `NewsItem`, повертає словник «мова → кількість».

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
def count_by_lang(items: list[NewsItem]) -> dict[str, int]:
    counts: dict[str, int] = {}
    for item in items:
        counts[item.lang] = counts.get(item.lang, 0) + 1
    return counts
# END SOLUTION


annotations = count_by_lang.__annotations__
assert count_by_lang([NewsItem.from_raw(raw_news[0])]) == {"ru": 1}
assert annotations["items"] == list[NewsItem] and annotations["return"] == dict[str, int]
print(annotations)
print("✅ Вправа 1 пройдена")

### Вправа 2

Клітинка записує файл `my_check.py` і запускає на ньому `mypy`. У коді дві помилки, які знайде mypy. Виправ код (не анотації!) так, щоб mypy сказав `Success`.

In [ ]:
source = """
from news_hub.models import NewsItem
from news_hub.parser import RawNews


def headline(item: RawNews) -> str:
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return item["title"].upper()
    # END SOLUTION


def hour(item: NewsItem) -> int | None:
    # YOUR CODE HERE
    # BEGIN SOLUTION
    if item.published_time is None:
        return None
    return item.published_time.hour
    # END SOLUTION
"""
# Помилки, які треба виправити: item["titel"] і item.published_time.hour без перевірки на None
open("my_check.py", "w", encoding="utf-8").write(source)
result = subprocess.run([sys.executable, "-m", "mypy", "--cache-dir", os.devnull, "my_check.py"],
                        capture_output=True, text=True)
print(result.stdout)
assert result.returncode == 0, "mypy ще знаходить помилки"
os.remove("my_check.py")
print("✅ Вправа 2 пройдена")

**🔮 Прогноз:** Що станеться, якщо виконати `item: RawNews = {'title': 42}` — Python кине помилку?

<details>
<summary>Відповідь</summary>

Ні. Анотації під час роботи не перевіряються: змінна отримає словник з числом. Помилку покаже лише mypy (до запуску) або Pydantic (якщо дані проходять через модель).

</details>

---
## 2. `NewsItem`: перевірка справжніх новин

`NewsItem.from_raw(raw)` → перевірена модель або `ValidationError`. `source`, `lang` і `category` модель виводить з URL.

### Вправа 3

Створи модель з першої новини знімка (`raw_news[0]`); збережи в `first`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
first = NewsItem.from_raw(raw_news[0])
# END SOLUTION

assert first.title == "Реформа ЗСУ" and first.published_time is None, "у цієї новини datetime порожній"
assert (first.source, first.lang, first.category) == ("rbc.ua", "ru", "Новини")
print(repr(first))
print("✅ Вправа 3 пройдена")

### Вправа 4

Спробуй створити модель із зіпсованих даних `broken`. Множину полів з помилками (`e["loc"][0]` для кожної помилки) збережи в `bad_fields`.

In [ ]:
broken: RawNews = {"title": "Коротко", "url": "https://example.com/ukr/news/x.html",
                   "category": "", "description": "", "datetime": "25:99"}
# YOUR CODE HERE
# BEGIN SOLUTION
try:
    NewsItem.from_raw(broken)
except ValidationError as error:
    bad_fields = {e["loc"][0] for e in error.errors()}
# END SOLUTION

assert bad_fields == {"title", "url", "published_time"}, "одна ValidationError збирає всі помилки"
print(bad_fields)
print("✅ Вправа 4 пройдена")

---
## 3. Власні моделі

### Вправа 5

Опиши модель `Headline`: `title` — рядок 10–300 символів, `lang` — лише `"uk"` або `"ru"`, `hour` — ціле 0–23 або `None` (за замовчуванням `None`).

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
class Headline(BaseModel):
    title: str = Field(min_length=10, max_length=300)
    lang: Literal["uk", "ru"]
    hour: int | None = Field(default=None, ge=0, le=23)
# END SOLUTION

assert Headline(title="Уряд затвердив бюджет", lang="uk").hour is None
assert Headline(title="Уряд затвердив бюджет", lang="ru", hour="14").hour == 14, "рядок '14' → число"
for bad in ({"title": "Коротко", "lang": "uk"}, {"title": "Уряд затвердив бюджет", "lang": "en"},
            {"title": "Уряд затвердив бюджет", "lang": "uk", "hour": 25}):
    try:
        Headline(**bad)
        raise AssertionError(f"мало не пройти: {bad}")
    except ValidationError:
        pass
print("✅ Вправа 5 пройдена")

### Вправа 6

Додай у `CalmHeadline` валідатор: заголовок з одних великих літер (`title.isupper()`) — помилка «заголовок-клікбейт».

In [ ]:
class CalmHeadline(BaseModel):
    title: str = Field(min_length=10)

    # YOUR CODE HERE
    # BEGIN SOLUTION
    @field_validator("title")
    @classmethod
    def not_shouting(cls, title: str) -> str:
        if title.isupper():
            raise ValueError("заголовок-клікбейт")
        return title
    # END SOLUTION


assert CalmHeadline(title="Уряд затвердив бюджет").title == "Уряд затвердив бюджет"
try:
    CalmHeadline(title="ТЕРМІНОВО!!! ШОК!!!")
    raise AssertionError("мало не пройти")
except ValidationError as error:
    print(error.errors()[0]["msg"])
print("✅ Вправа 6 пройдена")

---
## 4. Весь знімок, JSON і JSON Schema

### Вправа 7

Перевір увесь знімок функцією `validate_news`. Кількість перевірених — у `n_valid`, новин без часу публікації — у `n_no_time`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
valid, rejected = validate_news(raw_news)
n_valid = len(valid)
n_no_time = sum(item.published_time is None for item in valid)
# END SOLUTION

assert (n_valid, len(rejected), n_no_time) == (168, 0, 31)
print(count_by_lang(valid))
print("✅ Вправа 7 пройдена")

**🔮 Прогноз:** Яким буде `published_time` у `model_dump_json()` — `14:19` чи `14:19:00`?

<details>
<summary>Відповідь</summary>

`"14:19:00"`: у моделі це `datetime.time`, а JSON-формат часу — ISO з секундами.

</details>

### Вправа 8

Отримай JSON Schema моделі `NewsItem`; список обов'язкових полів — у `required`.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
required = NewsItem.model_json_schema()["required"]
# END SOLUTION

assert required == ["title", "url", "source", "lang", "category"], "published_time має значення за замовчуванням"
print(valid[1].model_dump_json())
print("✅ Вправа 8 пройдена")

### Вправа 9

`TypeAdapter` перевіряє типи без моделі. Перевір `["1", "2", "3"]` як `list[int]` (результат — у `numbers`) і переконайся, що `["1", "два"]` не проходить.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
adapter = TypeAdapter(list[int])
numbers = adapter.validate_python(["1", "2", "3"])
# END SOLUTION

assert numbers == [1, 2, 3]
try:
    adapter.validate_python(["1", "два"])
    raise AssertionError("мало не пройти")
except ValidationError as error:
    print(error.errors()[0]["loc"], error.errors()[0]["msg"])
print("✅ Вправа 9 пройдена")

---
## 5. Знайди помилку

Так `news_dashboard` старого курсу визначав розділ сайту:

```python
url_parts = [p for p in url.replace("https://www.rbc.ua", "").split("/") if p]
raw_cat = url_parts[1]
```

Для новини з `https://auto.rbc.ua/rus/news/…` «розділом» стає `auto.rbc.ua`.

### Вправа 10

Напиши `section_of(url)` через `urllib.parse.urlsplit`: друга частина **шляху** URL (`/rus/news/…` → `"news"`), незалежно від домену.

In [ ]:
from urllib.parse import urlsplit


def section_of(url: str) -> str:
    # YOUR CODE HERE
    # BEGIN SOLUTION
    path = [segment for segment in urlsplit(url).path.split("/") if segment]
    return path[1] if len(path) > 1 else ""
    # END SOLUTION


auto_url = next(raw["url"] for raw in raw_news if "auto.rbc.ua" in raw["url"])
assert section_of(auto_url) == "news", auto_url
assert section_of("https://www.rbc.ua/ukr/economic/hryvnia-2.html") == "economic"
assert section_of("https://www.rbc.ua/") == ""
print(auto_url, "→", section_of(auto_url))
print("✅ Вправа 10 пройдена")

---
## Самоперевірка

1. Що перевіряє `mypy`, а що — Pydantic?
2. Чим `TypedDict` кращий за `dict` для результату парсера?
3. Навіщо `published_time: time | None`?
4. Що дає `frozen=True`?
5. Чому URL розбирають `urlsplit`, а не `replace`/`split`?

<details>
<summary>Відповіді</summary>

1. `mypy` — код до запуску (типи, ключі, `None`); Pydantic — дані під час роботи.
2. mypy знає ключі: одруківка — помилка до запуску, а не `KeyError` у роботі.
3. У 31 новини часу немає; тип чесно це описує, і mypy змусить обробити `None`.
4. Перевірену модель не можна змінити присвоєнням — дані лишаються правильними.
5. `urlsplit` розбирає URL за стандартом (домен окремо від шляху); `replace` працює лише для точного збігу рядка.

</details>

## Далі

- Книга курсу: [Урок 36](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_36/) — рефакторинги «типи» і «Pydantic», архітектура межі довіри.
- У папці `news_hub`: `pytest` і `mypy --strict news_hub`.
- **Урок 37** — FastAPI над `NewsItem`: `/api/news`, `/docs`, Postman.